### Imports

In [15]:
import numpy as np

### Sigmoid activation and its derivative

Sigmoid function: $\sigma(x) = \frac{1}{1 + e^{-x}}$

Derivative: $\sigma'(x) = \frac{e^{-x}}{(1 + e^{-x})^2} = \sigma(x)\left(1 - \sigma(x)\right)$

In [16]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    sx = sigmoid(x)
    return sx * (1 - sx)

### Mean Squared Error and its derivative


Mean Squared Error (MSE) loss: $\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N}(\hat{y}_i - y_i)^2, N=4$

Derivative: $\frac{\partial \mathcal{L}}{\partial \hat{y}_i} = (\hat{y}_i - y_i)$

In [17]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def mse_derivative(y_true, y_pred):
    # N = 4
    # return (2 / N) * (y_pred - y_true)
    # const (2 / N) = 1/2 can be dropped by selecting the right learning rate
    return (y_pred - y_true)

### XOR training data


In [18]:
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])
y = np.array([[0], [1], [1], [0]])


In [19]:
mse(y, [[0], [0], [1], [0]])


0.25

In [20]:
mse_derivative(y, [[0], [0], [1], [0]])

array([[ 0],
       [-1],
       [ 0],
       [ 0]])

### Network parameters

In [21]:
# Fixed
input_size = 2
output_size = 1
N = X.shape[0]  # Number of training samples

# Can be tuned
hidden_size = 32 
learning_rate = 0.1
epochs = 10000

### Weight and bias initialization


In [57]:
np.random.seed(42)
W1 = np.random.randn(input_size, hidden_size)    # shape: (input_size, hidden_size)
b1 = np.zeros((1, hidden_size))                  # shape: (1, hidden_size)
W2 = np.random.randn(hidden_size, output_size)   # shape: (hidden_size, 1)
b2 = np.zeros((1, output_size))                  # shape: (1, output_size)

### Network

- Inputs: $x \in \mathbb{R}^2$
- Hidden layer ($H$ neurons): $z_1 = x W_1 + b_1$, $a_1 = \sigma(z_1)$; 
  - $W_1$: weights of hidden layer (shape $2 \times H$)
  - $b_1$: bias of the hidden layer (shape $1 \times H$)
  - $z_1$: weighted sum hidden layer (shape $4 \times H$)
  - $a_1$: activation output hidden layer (shape $4 \times H$)
- Output layer (1 neuron): $z_2 = a_1 W_2 + b_2$, $\hat{y} = a_2 = \sigma(z_2)$
  - $W_2$: weights of the output layer (shape $H \times 1$)
  - $b_2$: bias of the output layer (shape $1 \times 1$)
  - $z_2$: weighted sum output layer (shape $4 \times 1$)
  - $a_2$: activation output layer (shape $4 \times 1$)

- Loss: $\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N}(\hat{y}_i - y_i)^2$

### Gradients to compute: $\frac{\partial \mathcal{L}}{\partial W_2}, \frac{\partial 
\mathcal{L}}{\partial b_2}, \frac{\partial \mathcal{L}}{\partial W_1}, \frac{\partial \mathcal{L}}{\partial b_1}$

### Gradient computation by back propagation
$\frac{\partial \mathcal{L}}{\partial \hat{y}}=(\hat{y} - y)$ Loss to each $\hat{y}_i=a_2$ (prediction, output layer 2 for each input $x_i$)
 
 $\frac{\partial \mathcal{L}}{\partial z_2}=\frac{\partial \mathcal{L}}{\partial \hat{y}}\hat{y}(1 - \hat{y})$ Loss to $z_2$ (weighted sum layer 2) = Loss to $\hat{y}$ times $\hat{y}$ to $z_2$; $\hat{y} = \sigma(z_2)$ and $\sigma'(x)=\sigma(x)(1-\sigma(x))$ 

 $\frac{\partial \mathcal{L}}{\partial W_2}= a_1^T \cdot \frac{\partial \mathcal{L}}{\partial z_2}  $ Loss to $W_2$ (weights layer 2) = $z_2$ to $W_2$ dot loss to $z_2$; $z_2 = a_1 W_2 + b_2$
 
 $\frac{\partial \mathcal{L}}{\partial b_2} = \frac{\partial \mathcal{L}}{\partial z_2}$ Loss to $b_2$ (bias layer 2) = Loss to $z_2$; $z_2$ to $b_2=1$
 
 $\frac{\partial \mathcal{L}}{\partial a_1}=\frac{\partial \mathcal{L}}{\partial z_2} \cdot W_2^T$ Loss to $a_1$ (output layer 1) = Loss to $z_2$ times $z_2$ to $a_1$
 
 $\frac{\partial \mathcal{L}}{\partial z_1}=\frac{\partial \mathcal{L}}{\partial a_1} a_1(1 - a_1)$ Loss to $z_1$ (weighted sum layer 1) = Loss to $a_1$ dot $a_1$ to $z_1$; $a_1 = \sigma(z_1)$ 
 
 $\frac{\partial \mathcal{L}}{\partial W_1}=x^T \cdot \frac{\partial \mathcal{L}}{\partial z_1}$ Loss to $W_1$ (weights layer 1) = $z_1$ to $W_1$ dot loss to $z_1$; $z_1 = x W_1 + b_1$
 
 $\frac{\partial \mathcal{L}}{\partial b_1} = \frac{\partial \mathcal{L}}{\partial z_1}$ Loss to $b_1$ (biases layer 1) = Loss to $z_2$; $z_2$ to $b_2=1$
 

### Training loop

In [61]:
for epoch in range(epochs):
    # ---- Forward Pass ----
    z1 = np.dot(X, W1) + b1                     # shape: (4, hidden_size)
    a1 = sigmoid(z1)                            # shape: (4, hidden_size)

    z2 = np.dot(a1, W2) + b2                    # shape: (4, 1)
    a2 = sigmoid(z2)                            # shape: (4, 1)

    # ---- Loss Calculation ----
    loss = mse(y, a2)                           # scalar

    # ---- Backward Pass ----
    dL_da2 = mse_derivative(y, a2)              # shape: (4, 1)
    dL_dz2 = dL_da2 * sigmoid_derivative(z2)    # shape: (4, 1)

    # Gradients for W2 and b2
    dW2 = np.dot(a1.T, dL_dz2)                  # shape: (hidden_size, 1)
    db2 = np.sum(dL_dz2, axis=0, keepdims=True) # shape: (1, 1)

    # Propagate to hidden layer
    dL_da1 = np.dot(dL_dz2, W2.T)               # shape: (4, hidden_size)
    dL_dz1 = dL_da1 * sigmoid_derivative(z1)    # shape: (4, hidden_size)

    # Gradients for W1 and b1
    dW1 = np.dot(X.T, dL_dz1)                   # shape: (2, hidden_size)
    db1 = np.sum(dL_dz1, axis=0, keepdims=True) # shape: (1, hidden_size)

    # ---- Update Weights ----
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    # Print progress
    if epoch % 1000 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.6f}")

# ---- Final Prediction ----
print("\nFinal Predictions:")
print(np.round(a2, 3))


Epoch 0, Loss: 0.323704
Epoch 1000, Loss: 0.043212
Epoch 2000, Loss: 0.010601
Epoch 3000, Loss: 0.005127
Epoch 4000, Loss: 0.003211
Epoch 5000, Loss: 0.002283
Epoch 6000, Loss: 0.001748
Epoch 7000, Loss: 0.001404
Epoch 8000, Loss: 0.001168
Epoch 9000, Loss: 0.000995

Final Predictions:
[[0.026]
 [0.97 ]
 [0.97 ]
 [0.032]]
